# Pràctica amb ordinador 4. Filogènia mitocondrial d'hominins moderns i arcaics
## Preparació de l'ordinador
En esta pràctica utilitzarem els paquets `phangorn`, `Biostrings` i `pegas`. Si no estan intal·lats al teu ordinador, executa les ordres següents en un terminal:

> `install.packages('phangorn')`

> `install.packages('BiocManager')`

> `BiocManager::install('DECIPHER')`

> `install.packages('pegas')`

I a continuació, carrega'ls:

In [1]:
suppressMessages(library('phangorn'))
suppressMessages(library('DECIPHER'))
suppressMessages(library('pegas'))

## Introducció
Al web [http://www.phylotree.org](http://www.phylotree.org) s'acumulen més de 24000 seqüències completes d'ADN mitocondrial humà. Tota la diversitat genètica mitocondrial de les poblacions humanes actuals pot classificar-se en uns pocs haplotips principals, la filogènia dels quals es reprodueix a la figura següent:

![](phylotree.png)

L'objectiu d'aquesta pràctica és utilitzar dades d'ADN mitocondrial per estudiar les relacions filogenètiques entre humans actuals i alguns dels nostres parents més pròxims, com els neandertals i els denisovans. A més, aplicarem la teoria de la coalescència a les dades per extreure informació sobre les diferents poblacions d'hominins.

Les dades es troben a l'arxiu `mtDNA.fasta` i són un alineament dels cromosomes mitocondrials complets de 9 humans moderns, 19 neandertals, 4 denisovans, un *Homo heidelbergensis*, 4 bonobos (*Pan paniscus*), 4 ximpanzés (*Pan troglodytes*) i un goril·la (que ens servirà d'*outgroup*).

## Reconstrucció filogenètica i calibratge del rellotge molecular
Igual que en la pràctica anterior, el bloc de codi següent carrega les seqüències (ja alineades) en la sessió de treball i les mostra en una pestanya nova del navegador:

In [2]:
mtDNA <- readDNAStringSet('mtDNA.fasta')
BrowseSeqs(mtDNA, htmlFile = 'mtDNA.html', openURL = TRUE)

Per reconstruir la filogènia de les seqüències, podem estimar les distàncies genètiques entre elles amb un model d'evolució molecular i aplicar l'algoritme *neighbor joining*, que és eficient i suficientment acurat. Observa que la funció que estima les distàncies, `dist.dna()`, exigeix que transformem primer l'alineament en un objecte de tipus `DNAbin`:

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 12)
mtDNAb <- as.DNAbin(mtDNA)
distancies <- dist.dna(mtDNAb, model = 'K81')
arbre <- NJ(distancies)
plot(arbre)

El *neighbor-joining* produeix arbres *desarrelats*. Per visualitzar l'arbre més correctament podem indicar que la seqüència de goril·la (*G.gorilla_D38114*) és l'*outgroup*.

In [ ]:
arbre <- root(arbre, outgroup = 'G.gorilla_D38114', resolve.root = TRUE)
plot(arbre)
axisPhylo()

En arrelar l'arbre, les branques esdevenen *orientades*: el temps avança des de l'arrel cap a les puntes (o *fulles*). Però l'arbre no és encara *ultramètric*: la distància (nombre de substitucions per nucleòtid) des de l'arrel fins qulasevol de les fulles no és sempre la mateixa. Hi ha almenys dos bons motius perquè no siga ultramètric. En primer lloc, el punt exacte d'arrelament de l'arbre és arbitrari, dins la branca que separa l'*outgroup*, i la funció `root()` podia haver triat un punt més a la dreta, per aliniar millor el goril·la amb la resta de fulles de l'arbre. En segon lloc, no totes les seqüències són contemporànies i per tant no totes tenen per què haver acumulat la mateixa quantitat de canvis des de l'arrel. Podem intentar solucionar aquests dos problemes amb la funció `rtt()` del paquet `ape` (carregat automàticament amb `phangorn`). Al bloc següent, `rtt()` fixa l'arrel de l'arbre en la posició més compatible possible amb la hipòtesi del **rellotge molecular** i tenint en compte l'edat estimada de les seqüències, d'acord amb la informació arreplegada a l'apèndix.

In [ ]:
tipDates <- c(0, 0, 0, 0, 0, 0, 0, 0, 0,
              -400000,
              -39000, -50000, -110000, -110000,
              -39820, -40096, -38515, -38310, -39000, -40000, -41210,
              -42430, -42540, -43230, -43780, -44290, -44770, -49000,
              -50000, -82752, -65000, -122287, -110450,
              0, 0, 0, 0, 0, 0, 0, 0, 0)
              
arbre <- rtt(arbre, tip.dates = tipDates, objective = 'rms')
plot(arbre, x.lim = 0.08, label.offset = 0.001)
axisPhylo()

Les longituds de les branques de l'arbre representen el nombre estimat de canvis nucleotídics acumulats en cada branca. Si sumàrem les longituds de les branques que separen dues fulles, trobaríem un valor molt semblant a la distància estimada abans amb la funció `dist.dna()`.

Si els canvis nucleotídics s'acumularen exactament al mateix ritme en totes les branques (hipòtesi del rellotge molecular) i si les haguérem estimat sense error, aleshores les longituds de les branques serien *també* proporcionals al temps transcorregut, i podríem datar els ancestres (o nodes interns) a partir de la seua profunditat en l'arbre.

La funció `estimate.dates()` utilitza les dates conegudes per estimar les edats o dates de la resta de nodes d'un arbre arrelat. Per facilitar-li la feina, li proporcionem un vector amb les edats no només de les fulles de l'arbre (42 primers nodes), sinó també amb la de l'ancestre comú més recent entre humans i ximpanzés (node 44 de l'arbre): 6.5 milions d'anys, aproximadament.

In [13]:
nodeDates <- rep(NA, 83)
nodeDates[1:42] <- tipDates
nodeDates[44] <- -6500000
edatsEstimades <- estimate.dates(arbre, node.dates = nodeDates)
round(edatsEstimades,0)

[1]        0        0        0        0        0        0        0        0
 [9]        0  -400000   -39000   -50000  -110000  -110000   -39820   -40096
[17]   -38515   -38310   -39000   -40000   -41210   -42430   -42540   -43230
[25]   -43780   -44290   -44770   -49000   -50000   -82752   -65000  -122287
[33]  -110450        0        0        0        0        0        0        0
[41]        0        0 -8238553 -6508977 -1410125  -755591  -456008  -231428
[49]  -141285  -117552   -76396   -73775   -44770   -44770   -41210   -56839
[57]   -45267   -42540   -40096   -59241   -59153   -49167   -99540  -214977
[65]  -315884  -283114  -138044   -92828   -59190   -47362   -57354  -240511
[73] -1007486  -340257  -246368   -55051 -2845572 -1469764  -165022  -142008
[81]  -627921  -325857  -300249

In [ ]:
par(mar = c(5,5,0,0))
plot(arbre, x.lim = c(-0.0015, 0.08), label.offset = 0.001)
nodelabels(round(edatsEstimades[43:83],0),
           frame = 'none', cex = 0.7, adj = c(1.1, -0.2))
axisPhylo()

## Mida poblacional efectiva
Contesta les preguntes següents. Pots fer-ho sobre aquest mateix document, fent doble click al bloc de text corresponent.

1. Quina edat se li atribueix a l'ancestre comú més recent de les 9 mostres humanes? I a l'ancestre comú més recent de les mostres neandertals?

2. Assumint un temps de generació de 20 anys, quantes generacions enrere va viure l'ancestre comú més recent de les 9 mostres d'humans moderns?

3. Assumint el mateix temps de generació i suposant que els 19 neandertals van viure fa 53550 anys, quantes generacions els separen del seu ancestre comú més recent?

4. D'acord amb la teoria de la coalescència, l'ancestre comú més recent d'una mostra aleatòria d'*n* seqüències *haploides* d'una població de mida constant esperem trobar-lo, en promig, quasi tantes generacions enrere com dues vegades el nombre individus *efectius*. És a dir, $E(H_n)\approx 2N_e$. D'acord amb açò, quines serien les mides poblacionals efectives dels humans moderns, dels neandertals, dels denisovans, dels ximpanzés i dels bonobos?

## Taxa de mutació mitocondrial
La mida poblacional efectiva i la taxa de mutació determinen juntes la diversitat genètica d'una població, mitjançant el paràmetre $\theta = 4N_e\mu$, en diploides, o $\theta = 2N_e\mu$, en haploides. La diversitat la podem mesurar com $\pi$, la proporció mitjana de diferències nucleotídiques entre dues seqüències triades a l'atzar en la població, o com la proporció $s$ de posicions variables d'un alineament d'*n* seqüències aleatòriament triades en la població. Les esperances teòriques d'aquestes mesures són $E(\pi)=\theta$ i $E(s)=\theta\sum_{i=1}^{n-1}\frac{1}{i}$, on $\theta$ és $2N_e\mu$ en haploides.

El codi següent examina l'alineament original i extreu el nombre de llocs variables entre les seqüències d'humans moderns:

In [ ]:
length(seg.sites(mtDNAb[startsWith(names(mtDNAb), 'H.s.modern')]))

5. Seguint l'exemple del comandament anterior, troba el nombre de llocs segregants entre les seqüències de cada població.

6. Tenint en compte que l'alineament és de 16603 posicions, utilitza la proporció de llocs variables i les estimacions prèvies de mida poblacional per estimar la taxa de mutació mitocondrial en cada població.

El paràmetre $\pi$ de cada població es pot obtenir amb la funció `nuc.div()` del paquet `pegas`. Al bloc següent s'estima la $\pi$ dels ximpanzés:

In [ ]:
nuc.div(mtDNAb[startsWith(names(mtDNAb), 'P.troglodytes')])

7. Seguint el model, estima tu els valors de $\pi$ per a la resta de poblacions.

8. Estima de nou les taxes de mutació mitocondrials de cada població, però ara a partir dels valors estimats de $\pi$. Com es comparen els valors?

9. A partir de la datació dels nodes de l'arbre, també podem estimar una taxa de mutació mitjana, de tot l'arbre. Executa el comandament següent i veges com es compara amb les taxes de mutació que has estimat abans. Sabries dir en quines unitats estan les taxes de mutació?

In [ ]:
estimate.mu(arbre, nodeDates)

# Apèndix

 |   Espècie         |  Número d'accés |    Edat (anys) |   Referència          |  Nom                 |
 | ----------------- | --------------- | -------------- | --------------------- | -------------------- |
 | H.heidelbergensis |     NC_023100   |     400.000    | Meyer et al. 2014     | -                    |
 | H.s.denisova      |     NC_013993   |  30.000-48.000 | Krause et al. 2010    | Denisova 3 (falange) |
 | H.s.denisova      |     FR695060    |     >50.000?   | Reich et al. 2010     | Denisova 4 (molar)   |
 | H.s.denisova      |     KX663333    |     >100.000   | Slon et al. 2017      | Denisova 2 (dent)    |
 | H.s.denisova      |     KT780370    |     110.000?   | Sawyer et al. 2015    | Denisova 8 (molar)   |
 | H.s.neandertal    |     KX198085    |  32.697-46.942 | Posth et al. 2017     | GoyetQ374a-1         |
 | H.s.neandertal    |     KX198086    |  33.134-47.057 | Posth et al. 2017     | GoyetQ305-7          |
 | H.s.neandertal    |     MG025538    |  37.880-39.150 | Hajdinjak et al. 2018 | Spy-94a              |
 | H.s.neandertal    |     NC_011137   |      38.310    | Green et al. 2008     | Vindija 33.16        |
 | H.s.neandertal    |     FM865409    |      39.000    | Briggs et al. 2009    | Sidron 1253          |
 | H.s.neandertal    |     FM865407    |      40.000    | Briggs et al. 2009    | Feldhofer 1          |
 | H.s.neandertal    |     KX198088    |  40.620-41.800 | Rougier et al. 2016   | GoyetQ57-2           |
 | H.s.neandertal    |     KX198083    |  41.960-42.900 | Rougier et al. 2016   | GoyetQ57-3           |
 | H.s.neandertal    |     MG025540    |  42.080-43.000 | Hajdinjak et al. 2018 | GoyetQ56-1           |
 | H.s.neandertal    |     MG025536    |  42.720-43.740 | Hajdinjak et al. 2018 | Les Cottés Z4-1514   |
 | H.s.neandertal    |     MG025537    |  42.960-44.600 | Hajdinjak et al. 2018 | Mezmaiskaya2         |
 | H.s.neandertal    |     KX198087    |  43.430-45.150 | Rougier et al. 2016   | GoyetQ305-4          |
 | H.s.neandertal    |     KX198082    |  43.910-45.630 | Rougier et al. 2016   | GoyetQ57-1           |
 | H.s.neandertal    |     FM865408    |      49.000    | Briggs et al. 2009    | Feldhofer 2          |
 | H.s.neandertal    |     KU131206    |      >50.000   | Brown et al. 2016     | DC1227               |
 | H.s.neandertal    |     KF982693    | 56.213-109.290 | Posth et al. 2017     | Okladnikov2          |
 | H.s.neandertal    |     FM865411    |  60.000-70.000 | Briggs et al. 2009    | Mezmaiskaya1         |
 | H.s.neandertal    |     KY751400    | 62.013-182.560 | Posth et al. 2017     | HST                  |
 | H.s.neandertal    |     MK033602    | 90.900-130.000 | Douka et al. 2019     | Denisova15           |

## Bibliography

- Briggs, A.W. et al. 2009. *Science* 325(5938):318-21. [10.1126/science.1174462](https://doi.org/10.1126/science.1174462)
- Brown, S. et al. 2016. *Sci. Rep.* 6:23559. [10.1038/srep23559](https://doi.org/10.1038/srep23559)
- Douka, K. et al. 2019. *Nature* 565:640-644. [10.1038/s41586-018-0870-z](https://doi.org/10.1038/s41586-018-0870-z)
- Green, R.E. et al. 2008. *Cell* 134(3):416-426. [10.1016/j.cell.2008.06.021](https://doi.org/10.1016/j.cell.2008.06.021)
- Hajdinjak, M. et al. 2018. *Nature* 555(7698):652-656. [10.1038/nature26151](https://doi.org/10.1038/nature26151)
- Krause, J. et al.2010. *Nature* 464(7290):894-897 [10.1038/nature08976](https://doi.org/10.1038/nature08976)
- Meyer, M. et al. 2014. *Nature* 505(7483):403-406. [10.1038/nature12788](https://doi.org/10.1038/nature12788)
- Posth, C. et al. 2017. *Nat. Commun.* 8:16046. [10.1038/ncomms16046](https://doi.org/10.1038/ncomms16046)
- Reich, D. et al. 2010. *Nature* 468(7327):1053-60. [10.1038/nature09710](https://doi.org/10.1038/nature09710)
- Rougier, H. et al. 2016. *Sci. Rep.* 6:29005. [10.1038/srep29005](https://doi.org/10.1038/srep29005)
- Skoglund, P. et al. 2014. *Proc. Natl. Acad. Sci. U.S.A* 111(6):2229-2234. [10.1073%2Fpnas.1318934111](https://doi.org/10.1073%2Fpnas.1318934111)
- Sawyer, S. et al. 2015. *Proc. Natl. Acad. Sci. U.S.A.* 112(51):15696-15700. [10.1073/pnas.1519905112](https://doi.org/10.1073/pnas.1519905112)
- Slon, V. et al. 2017. *Sci. Adv.* 3:e1700186. [10.1126/sciadv.1700186](https://doi.org/10.1126/sciadv.1700186)
